# Feature Engineering: дефицитные анемии

**Вход:** `data/raw/deficiency_anemia.csv` (840 × 48)
**Выход:** `data/processed/df_features.csv` — с производными признаками

## Что создаём
1. **Базовые фичи:** anemia_by_WHO, inflammation_flag, sex_M, TSST_calc
2. **Флаги отклонений:** low_MCV, high_RDW, low_ferritin, low_B12, ...
3. **Флаги пропуска:** ferritin_measured, vitamin_B12_measured, ...

## Зачем
- Клинически осмысленные признаки → модель учится лучше
- Флаги пропуска → XGBoost понимает, что признак не измерялся

In [12]:
# ============ ИМПОРТЫ ============
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

RANDOM_STATE = 42

print(f"CWD: {os.getcwd()}")
print("Импорты загружены")

CWD: C:\Users\USER\Documents\67-git-kittens\notebooks
Импорты загружены


In [13]:
# ============ ЗАГРУЗКА ============
df = pd.read_csv('../data/raw/deficiency_anemia.csv')

print(f"Размер: {df.shape}")
print(f"Строк: {df.shape[0]}, Колонок: {df.shape[1]}")

Размер: (840, 48)
Строк: 840, Колонок: 48


## Блок 1: Базовые фичи

- `anemia_by_WHO` — анемия по критериям ВОЗ (бинарный)
- `inflammation_flag` — признаки воспаления (CRP > 5 или ESR > 20)
- `sex_M` — пол как бинарный (M=1, F=0)
- `TSAT_calc` — насыщение трансферрина, если пропущено в данных
- `age_group` — возрастная группа

In [14]:
# ============ БАЗОВЫЕ ФИЧИ ============

# 1. Анемия по ВОЗ (полоспецифичные пороги)
df['anemia_by_WHO'] = np.where(
    df['sex'] == 'F',
    (df['hemoglobin'] < 120).astype(int),
    (df['hemoglobin'] < 130).astype(int)
)

# 2. Флаг воспаления (CRP > 5 ИЛИ ESR > 20)
df['inflammation_flag'] = (
    (df['CRP'].fillna(0) > 5) | (df['ESR'].fillna(0) > 20)
).astype(int)

# 3. Пол как бинарный
df['sex_M'] = (df['sex'] == 'M').astype(int)

# 4. TSAT_calc — если TSAT пропущен, считаем из serum_iron / TIBC
df['TSAT_calc'] = df['TSAT'].copy()
mask = df['TSAT_calc'].isna() & df['serum_iron'].notna() & df['TIBC'].notna()
df.loc[mask, 'TSAT_calc'] = df.loc[mask, 'serum_iron'] / df.loc[mask, 'TIBC'] * 100

# 5. Возрастные группы
df['age_group'] = pd.cut(
    df['age_years'],
    bins=[17, 29, 39, 49, 59, 69, 120],
    labels=['18-29', '30-39', '40-49', '50-59', '60-69', '70+']
)

# Проверка
print("Создано 5 базовых фич:")
print(f"  anemia_by_WHO: {df['anemia_by_WHO'].sum()} анемиков")
print(f"  inflammation_flag: {df['inflammation_flag'].sum()} с воспалением")
print(f"  sex_M: {df['sex_M'].sum()} мужчин")
print(f"  TSAT_calc: заполнено {df['TSAT_calc'].notna().sum()} из 840")
print(f"  age_group: {df['age_group'].nunique()} групп")

Создано 5 базовых фич:
  anemia_by_WHO: 531 анемиков
  inflammation_flag: 153 с воспалением
  sex_M: 244 мужчин
  TSAT_calc: заполнено 540 из 840
  age_group: 6 групп


### Результаты Блока 1

| Фича | Значение | Комментарий |
|---|---|---|
| `anemia_by_WHO` | **531** (63%) | Совпадает с `df['anemia'].sum()` — правило ВОЗ корректно |
| `inflammation_flag` | **153** (18%) | CRP>5 или ESR>20 |
| `sex_M` | **244** (29%) | Дисбаланс пола: F=596 (71%), M=244 (29%) |
| `TSAT_calc` | **540** (64%) | Не восстановил новых — у 300 пациентов пропущены TSAT+serum_iron+TIBC вместе |
| `age_group` | **6** групп | 18-29, 30-39, 40-49, 50-59, 60-69, 70+ |

**Наблюдения:**
- Дисбаланс пола **клинически объясним** — женщины чаще сдают анализы на анемию.
- `TSAT_calc` **не вредит**, но и не даёт новых значений.
- `anemia_by_WHO` **полностью совпадает** с target `anemia` — правило ВОЗ работает корректно.

## Очистка: проверка дубликатов и несоответствий

- Дубликаты по `patient_id` — не должно быть
- Дубликаты по всем колонкам — не должно быть
- Несоответствия `anemia` vs правило ВОЗ — не должно быть
- Возраст, пол, Hb — в физиологических диапазонах

In [15]:
# ============ ПРОВЕРКА ДУБЛИКАТОВ ============
print("=== Дубликаты ===")
print(f"По patient_id: {df['patient_id'].duplicated().sum()}")
print(f"По всем колонкам: {df.duplicated().sum()}")

=== Дубликаты ===
По patient_id: 0
По всем колонкам: 0


In [16]:
# ============ ПРОВЕРКА НЕСООТВЕТСТВИЙ ============
print("=== Несоответствия ===")

# 1. anemia vs правило ВОЗ
print(f"1. anemia vs anemia_by_WHO: {(df['anemia'] != df['anemia_by_WHO']).sum()}")

# 2. Анемия при нормальном Hb
mask = (df['anemia'] == 1) & (
    ((df['sex'] == 'F') & (df['hemoglobin'] >= 120)) |
    ((df['sex'] == 'M') & (df['hemoglobin'] >= 130))
)
print(f"2. Анемия при нормальном Hb: {mask.sum()}")

# 3. Нет анемии при низком Hb
mask = (df['anemia'] == 0) & (
    ((df['sex'] == 'F') & (df['hemoglobin'] < 120)) |
    ((df['sex'] == 'M') & (df['hemoglobin'] < 130))
)
print(f"3. Нет анемии при низком Hb: {mask.sum()}")

# 4. Возраст вне 18-100
print(f"4. Возраст вне 18-100: {((df['age_years'] < 18) | (df['age_years'] > 100)).sum()}")

# 5. Пол не M/F
print(f"5. Пол не M/F: {(~df['sex'].isin(['M', 'F'])).sum()}")

# 6. Hb вне 30-220
print(f"6. Hb вне 30-220: {((df['hemoglobin'] < 30) | (df['hemoglobin'] > 220)).sum()}")

=== Несоответствия ===
1. anemia vs anemia_by_WHO: 0
2. Анемия при нормальном Hb: 0
3. Нет анемии при низком Hb: 0
4. Возраст вне 18-100: 0
5. Пол не M/F: 0
6. Hb вне 30-220: 0


### Результат очистки

- Дубликатов: **0**
- Несоответствий: **0**
- Возраст, пол, Hb — в норме

**Вывод:** данные чистые, можно создавать флаги на этих данных.

## Блок 2: Флаги клинических отклонений

Бинарные признаки: 1 — есть отклонение, 0 — норма или пропуск.

**Источники порогов** (согласовано с медэкспертом команды):
- КР МЗ РФ «Железодефицитная анемия» (2024)
- КР МЗ РФ «B12-дефицитная анемия» (2024)
- КР МЗ РФ «Фолиеводефицитная анемия» (2024)
- Сборник лабораторных норм центра гематологии
- 323-ФЗ

| Флаг | Порог |
|---|---|
| `low_MCV` | < 80 фл |
| `high_MCV` | > 100 фл |
| `high_RDW` | > 15% |
| `low_MCH` | < 27 пг |
| `high_MCH` | > 34 пг |
| `low_ferritin` | < 30 нг/мл |
| `very_low_ferritin` | < 15 нг/мл |
| `low_serum_iron` | < 10 (Ж) / < 12 (М) мкмоль/л |
| `low_transferrin` | < 2.0 г/л |
| `high_TIBC` | > 70 мкмоль/л |
| `low_TSAT` | < 20% |
| `high_sTfR` | > 4.6 мг/л |
| `low_RetHe` | < 28 пг |
| `low_B12` | < 200 пг/мл |
| `low_folate` | < 4 нг/мл |
| `low_B6` | < 20 нг/мл |
| `high_MMA` | > 0.4 мкмоль/л |
| `high_homocysteine` | > 15 мкмоль/л |
| `low_copper` | < 70 мкг/дл |
| `low_eGFR` | < 60 |
| `high_TSH` | > 4 мМЕ/л |

In [17]:
# ============ ФЛАГИ ОТКЛОНЕНИЙ ============

# Морфология эритроцитов
df['low_MCV'] = (df['MCV'] < 80).astype(int)
df['high_MCV'] = (df['MCV'] > 100).astype(int)
df['high_RDW'] = (df['RDW'] > 15).astype(int)
df['low_MCH'] = (df['MCH'] < 27).astype(int)
df['high_MCH'] = (df['MCH'] > 34).astype(int)

# Железо
df['low_ferritin'] = (df['ferritin'] < 30).astype(int)
df['very_low_ferritin'] = (df['ferritin'] < 15).astype(int)
df['low_serum_iron'] = np.where(
    df['sex'] == 'F',
    (df['serum_iron'] < 10).astype(int),
    (df['serum_iron'] < 12).astype(int)
)
df['low_transferrin'] = (df['transferrin'] < 2.0).astype(int)
df['high_TIBC'] = (df['TIBC'] > 70).astype(int)
df['low_TSAT'] = (df['TSAT_calc'] < 20).astype(int)
df['high_sTfR'] = (df['sTfR'] > 4.6).astype(int)
df['low_RetHe'] = (df['Ret_He'] < 28).astype(int)

# B12 / фолаты / B6
df['low_B12'] = (df['vitamin_B12'] < 200).astype(int)
df['low_folate'] = (df['folate'] < 4).astype(int)
df['low_B6'] = (df['vitamin_B6'] < 20).astype(int)
df['high_MMA'] = (df['MMA'] > 0.4).astype(int)
df['high_homocysteine'] = (df['homocysteine'] > 15).astype(int)

# Микроэлементы
df['low_copper'] = (df['copper'] < 70).astype(int)

# Соматический профиль
df['low_eGFR'] = (df['eGFR'] < 60).astype(int)
df['high_TSH'] = (df['TSH'] > 4).astype(int)

# Проверка
flag_cols = ['low_MCV', 'high_MCV', 'high_RDW', 'low_MCH', 'high_MCH',
             'low_ferritin', 'very_low_ferritin', 'low_serum_iron', 'low_transferrin',
             'high_TIBC', 'low_TSAT', 'high_sTfR', 'low_RetHe',
             'low_B12', 'low_folate', 'low_B6', 'high_MMA', 'high_homocysteine',
             'low_copper', 'low_eGFR', 'high_TSH']

print(f"Создано {len(flag_cols)} флагов отклонений:\n")
for col in flag_cols:
    count = df[col].sum()
    pct = count / len(df) * 100
    print(f"  {col:22s}: {count:3d} ({pct:4.1f}%)")

Создано 21 флагов отклонений:

  low_MCV               : 104 (12.4%)
  high_MCV              : 131 (15.6%)
  high_RDW              : 463 (55.1%)
  low_MCH               : 261 (31.1%)
  high_MCH              :  85 (10.1%)
  low_ferritin          : 233 (27.7%)
  very_low_ferritin     : 144 (17.1%)
  low_serum_iron        : 243 (28.9%)
  low_transferrin       :  34 ( 4.0%)
  high_TIBC             : 259 (30.8%)
  low_TSAT              : 304 (36.2%)
  high_sTfR             :  15 ( 1.8%)
  low_RetHe             :  85 (10.1%)
  low_B12               : 129 (15.4%)
  low_folate            : 130 (15.5%)
  low_B6                :  24 ( 2.9%)
  high_MMA              :  90 (10.7%)
  high_homocysteine     : 158 (18.8%)
  low_copper            : 124 (14.8%)
  low_eGFR              :  47 ( 5.6%)
  high_TSH              :  27 ( 3.2%)


### Результаты Блока 2: Флаги отклонений

Создано **21 флаг**. Распределение:

| Флаг | Кол-во | % |
|---|---|---|
| `high_RDW` | 463 | 55.1% |
| `low_TSAT` | 304 | 36.2% |
| `low_MCH` | 261 | 31.1% |
| `high_TIBC` | 259 | 30.8% |
| `low_serum_iron` | 243 | 28.9% |
| `low_ferritin` | 233 | 27.7% |
| `high_homocysteine` | 158 | 18.8% |
| `very_low_ferritin` | 144 | 17.1% |
| `high_MCV` | 131 | 15.6% |
| `low_folate` | 130 | 15.5% |
| `low_B12` | 129 | 15.4% |
| `low_copper` | 124 | 14.8% |
| `low_MCV` | 104 | 12.4% |
| `high_MMA` | 90 | 10.7% |
| `high_MCH` | 85 | 10.1% |
| `low_RetHe` | 85 | 10.1% |
| `low_eGFR` | 47 | 5.6% |
| `low_transferrin` | 34 | 4.0% |
| `high_TSH` | 27 | 3.2% |
| `low_B6` | 24 | 2.9% |
| `high_sTfR` | 15 | 1.8% |

**Проверка распределений (по заполненным значениям):**

| Признак | Порог | Заполнено | > Порога | Оценка |
|---|---|---|---|---|
| `sTfR` | > 4.6 | 252 | 15 (6%) | ✅ 95-й перцентиль = 4.70 — порог корректен |
| `transferrin` | < 2.0 | 540 | 34 (6%) | ✅ Нижние 6% — тяжёлые случаи |
| `TIBC` | > 70 | 540 | 259 (48%) | ✅ Median 69.4 — половина выше |
| `vitamin_B6` | < 20 | 135 | 24 (18%) | ✅ B6-дефицит редкий |
| `TSH` | > 4 | 425 | 27 (6%) | ✅ Гипотиреоз редкий |

**Все пороги корректны.** Малое количество срабатываний по редким дефицитам — ожидаемо.

## Блок 3: Флаги пропуска

XGBoost/LightGBM могут использовать **факт отсутствия** признака как сигнал:
если ферритин не измеряли — врач не подозревал ЖД.

Создаём бинарные флаги `*_measured` для ключевых признаков.

**Логика:** 1 — признак измерен, 0 — пропущен (NaN).

In [18]:
# ============ ФЛАГИ ПРОПУСКА ============
measured_cols = [
    'ferritin', 'vitamin_B12', 'TSAT', 'sTfR', 'Ret_He',
    'active_B12', 'MMA', 'homocysteine', 'folate', 'vitamin_B6',
    'copper', 'ceruloplasmin', 'TSH', 'CRP', 'ESR'
]

for col in measured_cols:
    df[f'{col}_measured'] = df[col].notna().astype(int)

# Проверка
measured_flags = [f'{col}_measured' for col in measured_cols]
print(f"Создано {len(measured_flags)} флагов пропуска:\n")
for col in measured_flags:
    count = df[col].sum()
    pct = count / len(df) * 100
    print(f"  {col:25s}: {count:3d} ({pct:4.1f}%)")

Создано 15 флагов пропуска:

  ferritin_measured        : 594 (70.7%)
  vitamin_B12_measured     : 513 (61.1%)
  TSAT_measured            : 540 (64.3%)
  sTfR_measured            : 252 (30.0%)
  Ret_He_measured          : 226 (26.9%)
  active_B12_measured      : 252 (30.0%)
  MMA_measured             : 272 (32.4%)
  homocysteine_measured    : 298 (35.5%)
  folate_measured          : 455 (54.2%)
  vitamin_B6_measured      : 135 (16.1%)
  copper_measured          : 124 (14.8%)
  ceruloplasmin_measured   : 131 (15.6%)
  TSH_measured             : 425 (50.6%)
  CRP_measured             : 496 (59.0%)
  ESR_measured             : 492 (58.6%)


## Финальная проверка датасета

Проверяем итоговый размер и состав признаков перед сохранением.

In [19]:
# ============ ФИНАЛЬНАЯ ПРОВЕРКА ============
print(f"Итоговый размер: {df.shape}")
print(f"Строк: {df.shape[0]}, Колонок: {df.shape[1]}")
print(f"\nБыло изначально: 48 колонок")
print(f"Стало: {df.shape[1]} колонок")
print(f"Новых признаков: {df.shape[1] - 48}")

# Список новых колонок
new_cols = [c for c in df.columns if c not in pd.read_csv('../data/raw/deficiency_anemia.csv').columns]
print(f"\nНовые колонки ({len(new_cols)}):")
for c in new_cols:
    print(f"  - {c}")

Итоговый размер: (840, 89)
Строк: 840, Колонок: 89

Было изначально: 48 колонок
Стало: 89 колонок
Новых признаков: 41

Новые колонки (41):
  - anemia_by_WHO
  - inflammation_flag
  - sex_M
  - TSAT_calc
  - age_group
  - low_MCV
  - high_MCV
  - high_RDW
  - low_MCH
  - high_MCH
  - low_ferritin
  - very_low_ferritin
  - low_serum_iron
  - low_transferrin
  - high_TIBC
  - low_TSAT
  - high_sTfR
  - low_RetHe
  - low_B12
  - low_folate
  - low_B6
  - high_MMA
  - high_homocysteine
  - low_copper
  - low_eGFR
  - high_TSH
  - ferritin_measured
  - vitamin_B12_measured
  - TSAT_measured
  - sTfR_measured
  - Ret_He_measured
  - active_B12_measured
  - MMA_measured
  - homocysteine_measured
  - folate_measured
  - vitamin_B6_measured
  - copper_measured
  - ceruloplasmin_measured
  - TSH_measured
  - CRP_measured
  - ESR_measured


## Сохранение обработанного датасета

Сохраняем в `data/processed/df_features.csv` — для обучения моделей.

In [20]:
# ============ СОХРАНЕНИЕ ============
OUTPUT_PATH = '../data/processed/df_features.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f"Сохранено: {OUTPUT_PATH}")
print(f"Размер файла: {df.shape}")

# Проверка — загружаем обратно
df_check = pd.read_csv(OUTPUT_PATH)
print(f"\nПроверка загрузки: {df_check.shape}")
print(f"Совпадение: {df.shape == df_check.shape}")

Сохранено: ../data/processed/df_features.csv
Размер файла: (840, 89)

Проверка загрузки: (840, 89)
Совпадение: True


## Итоги Feature Engineering

**Создано:**
- 5 базовых фич: `anemia_by_WHO`, `inflammation_flag`, `sex_M`, `TSAT_calc`, `age_group`
- 21 флаг отклонений: `low_MCV`, `high_RDW`, `low_ferritin`, ...
- 15 флагов пропуска: `ferritin_measured`, `vitamin_B12_measured`, ...

**Итого:** 41 новый признак.

**Выходной файл:** `data/processed/df_features.csv`

**Следующий этап:** препроцессинг (train/test split) → модели (сокомандник).